In [1]:
#CO2-AT1
words = ["connected", "connecting", "connection"]

rules = {
    "ed": ("connect", "ed", "Inflectional"),
    "ing": ("connect", "ing", "Inflectional"),
    "ion": ("connect", "ion", "Derivational")
}

print(f"{'Word':<15}{'Root':<15}{'Suffix':<10}{'Type':<18}{'Normalized'}")
print("-" * 70)

for word in words:
    for suffix, (root, suf, typ) in rules.items():
        if word.endswith(suffix):
            print(f"{word:<15}{root:<15}{suf:<10}{typ:<18}{root}")
            break


Word           Root           Suffix    Type              Normalized
----------------------------------------------------------------------
connected      connect        ed        Inflectional      connect
connecting     connect        ing       Inflectional      connect
connection     connect        ion       Derivational      connect


In [2]:
#CO2-AT1
words = ["unhappy", "happiness", "happily"]

rules = {
    "unhappy": ("un", "happy", "", "Derivational"),
    "happiness": ("", "happy", "ness", "Derivational"),
    "happily": ("", "happy", "ly", "Derivational")
}

print(f"{'Word':<15}{'Prefix':<10}{'Base':<10}{'Suffix':<10}{'Type':<18}{'Root'}")
print("-" * 80)

for word in words:
    prefix, base, suffix, typ = rules[word]
    print(f"{word:<15}{prefix:<10}{base:<10}{suffix:<10}{typ:<18}{base}")


Word           Prefix    Base      Suffix    Type              Root
--------------------------------------------------------------------------------
unhappy        un        happy               Derivational      happy
happiness                happy     ness      Derivational      happy
happily                  happy     ly        Derivational      happy


In [ ]:
#CO2-AT1
words = ["played", "player", "playing"]

rules = {
    "ed": ("play", "ed", "Inflectional"),
    "er": ("play", "er", "Derivational"),
    "ing": ("play", "ing", "Inflectional")
}

print(f"{'Word':<15}{'Stem':<15}{'Removed Affix':<15}{'Type':<18}{'Normalized'}")
print("-" * 85)

for word in words:
    for suffix, (stem, affix, typ) in rules.items():
        if word.endswith(suffix):
            print(f"{word:<15}{stem:<15}{affix:<15}{typ:<18}{stem}")
            break


In [ ]:
#CO2-AT1
words = ["writes", "writing", "written"]

transitions = {
    "writes": ["q0", "q1", "q2"],
    "writing": ["q0", "q1", "q3"],
    "written": ["q0", "q4"]
}

analysis = {
    "writes": ("write + s", "write", "Regular Inflection"),
    "writing": ("write + ing", "write", "Regular Inflection"),
    "written": ("write + en", "write", "Irregular Inflection")
}

print(f"{'Word':<12}{'State Path':<20}{'Morphology':<18}{'Root':<12}{'Type'}")
print("-" * 85)

for word in words:
    morphology, root, typ = analysis[word]
    path = " -> ".join(transitions[word])
    print(f"{word:<12}{path:<20}{morphology:<18}{root:<12}{typ}")


In [ ]:
#CO2-AT1
def porter_stem(word):
    steps = []

    if word.endswith("ational"):
        word = word[:-7] + "ate"
        steps.append("ATIONAL -> ATE: " + word)
    elif word.endswith("ation"):
        word = word[:-5] + "ate"
        steps.append("ATION -> ATE: " + word)

    if word.endswith("e") and len(word) > 3:
        word = word[:-1]
        steps.append("E -> removed: " + word)

    return steps, word


words = ["relational", "relation", "relate"]

print(f"{'Word':<15}{'Applied Rules / Intermediate Forms':<45}{'Final Stem'}")
print("-" * 85)

for word in words:
    steps, stem = porter_stem(word)
    print(f"{word:<15}{' | '.join(steps):<45}{stem}")


In [4]:
#CO2_AT2
corpus = [
    [("The", "DT"), ("boy", "NN"), ("eats", "VBZ"), ("rice", "NN")],
    [("The", "DT"), ("girl", "NN"), ("drinks", "VBZ"), ("milk", "NN")],
    [("A", "DT"), ("cat", "NN"), ("drinks", "VBZ"), ("milk", "NN")],
    [("The", "DT"), ("dog", "NN"), ("chases", "VBZ"), ("cat", "NN")],
    [("A", "DT"), ("teacher", "NN"), ("teaches", "VBZ"), ("students", "NNS")],
    [("Students", "NNS"), ("study", "VBP"), ("English", "NN")],
    [("Birds", "NNS"), ("fly", "VBP"), ("high", "RB")],
    [("Children", "NNS"), ("play", "VBP"), ("games", "NNS")]
]

tags = ["DT", "NN", "VBZ", "NNS", "VBP", "RB"]

tag_count = {}
word_tag_count = {}
transition_count = {}
transition_total = {}
initial_count = {}

for sentence in corpus:
    first_tag = sentence[0][1]
    initial_count[first_tag] = initial_count.get(first_tag, 0) + 1

    for word, tag in sentence:
        tag_count[tag] = tag_count.get(tag, 0) + 1
        key = (word, tag)
        word_tag_count[key] = word_tag_count.get(key, 0) + 1

    for i in range(len(sentence) - 1):
        t1 = sentence[i][1]
        t2 = sentence[i + 1][1]

        key = (t1, t2)
        transition_count[key] = transition_count.get(key, 0) + 1
        transition_total[t1] = transition_total.get(t1, 0) + 1

emission = {}

for (word, tag), count in word_tag_count.items():
    emission[(word, tag)] = count / tag_count[tag]

transition = {}

for (t1, t2), count in transition_count.items():
    transition[(t1, t2)] = count / transition_total[t1]

initial = {}

for tag in tags:
    initial[tag] = initial_count.get(tag, 0) / len(corpus)

sentence = ["The", "cat", "drinks", "milk"]

viterbi = [{}]
backpointer = [{}]

for tag in tags:
    viterbi[0][tag] = initial[tag] * emission.get((sentence[0], tag), 0)
    backpointer[0][tag] = None

for i in range(1, len(sentence)):
    viterbi.append({})
    backpointer.append({})

    for current_tag in tags:
        best_probability = 0
        best_previous = None

        emit = emission.get((sentence[i], current_tag), 0)

        for previous_tag in tags:
            trans = transition.get((previous_tag, current_tag), 0)

            probability = (
                viterbi[i - 1][previous_tag]
                * trans
                * emit
            )

            if probability > best_probability:
                best_probability = probability
                best_previous = previous_tag

        viterbi[i][current_tag] = best_probability
        backpointer[i][current_tag] = best_previous

best_tag = None
best_probability = 0

for tag in tags:
    if viterbi[-1][tag] > best_probability:
        best_probability = viterbi[-1][tag]
        best_tag = tag

result = [best_tag]

for i in range(len(sentence) - 1, 0, -1):
    result.append(backpointer[i][result[-1]])

result.reverse()

print("Sentence:", " ".join(sentence))
print("POS Tags:", result)
print("Probability:", best_probability)


Sentence: The cat drinks milk
POS Tags: ['DT', 'NN', 'VBZ', 'NN']
Probability: 0.004800000000000002


In [5]:
#CO2_AT3.1

import re
import pandas as pd
from pathlib import Path

words = [
    "treatment",
    "treatable",
    "retreatment",
    "treated",
    "untreated"
]

morphology = {
    "treatment": {
        "prefix": "",
        "root": "treat",
        "suffix": "ment",
        "affix_type": "derivational"
    },
    "treatable": {
        "prefix": "",
        "root": "treat",
        "suffix": "able",
        "affix_type": "derivational"
    },
    "retreatment": {
        "prefix": "re",
        "root": "treat",
        "suffix": "ment",
        "affix_type": "derivational"
    },
    "treated": {
        "prefix": "",
        "root": "treat",
        "suffix": "ed",
        "affix_type": "inflectional"
    },
    "untreated": {
        "prefix": "un",
        "root": "treat",
        "suffix": "ed",
        "affix_type": "derivational + inflectional"
    }
}

incorrect_analyses = {
    "treatment": ("treat", "treat + ment"),
    "treatable": ("treat", "treat + able"),
    "retreatment": ("treat", "re + treat + ment"),
    "treated": ("treat", "treat + ed"),
    "untreated": ("treat", "un + treat + ed")
}

def analyze_word(word):
    if word in morphology:
        m = morphology[word]
        return {
            "Word": word,
            "Prefix": m["prefix"] if m["prefix"] else "NONE",
            "Root": m["root"],
            "Suffix": m["suffix"] if m["suffix"] else "NONE",
            "Affix Type": m["affix_type"]
        }

    return {
        "Word": word,
        "Prefix": "UNKNOWN",
        "Root": word,
        "Suffix": "UNKNOWN",
        "Affix Type": "UNKNOWN"
    }

results = pd.DataFrame([analyze_word(w) for w in words])

print("\nCORRECTED MORPHOLOGICAL ANALYSIS")
print(results.to_string(index=False))

print("\nORIGINAL VS CORRECTED")

for word in words:
    original = incorrect_analyses[word][0]
    corrected = incorrect_analyses[word][1]

    print(f"\nWord: {word}")
    print(f"Original Analysis : {original}")
    print(f"Corrected Analysis: {corrected}")

pubmed_url = (
    "https://raw.githubusercontent.com/Franck-Dernoncourt/"
    "PubMed-RCT/master/PubMed_20k_RCT/train.txt"
)

try:
    df = pd.read_csv(
        pubmed_url,
        sep="\t",
        header=None,
        names=["label", "text"],
        quoting=3
    )

    biomedical_text = " ".join(df["text"].astype(str).head(1000))

    corpus_matches = {
        word: len(re.findall(rf"\b{word}\b", biomedical_text.lower()))
        for word in words
    }

    print("\nPUBMED CORPUS OCCURRENCES")
    for word, count in corpus_matches.items():
        print(f"{word}: {count}")

except Exception as e:
    print("\nDataset could not be loaded:", e)

print("\nCORRECTED MORPHOLOGICAL STRATEGY")
print("1. Tokenize biomedical text.")
print("2. Detect known prefixes.")
print("3. Detect known suffixes.")
print("4. Preserve derivational affixes.")
print("5. Apply inflectional analysis separately.")
print("6. Validate the root using a biomedical lexicon.")
print("7. Store prefix, root and suffix as separate morphological features.")


CORRECTED MORPHOLOGICAL ANALYSIS
       Word Prefix  Root Suffix                  Affix Type
  treatment   NONE treat   ment                derivational
  treatable   NONE treat   able                derivational
retreatment     re treat   ment                derivational
    treated   NONE treat     ed                inflectional
  untreated     un treat     ed derivational + inflectional

ORIGINAL VS CORRECTED

Word: treatment
Original Analysis : treat
Corrected Analysis: treat + ment

Word: treatable
Original Analysis : treat
Corrected Analysis: treat + able

Word: retreatment
Original Analysis : treat
Corrected Analysis: re + treat + ment

Word: treated
Original Analysis : treat
Corrected Analysis: treat + ed

Word: untreated
Original Analysis : treat
Corrected Analysis: un + treat + ed

PUBMED CORPUS OCCURRENCES
treatment: 100
treatable: 0
retreatment: 0
treated: 11
untreated: 0

CORRECTED MORPHOLOGICAL STRATEGY
1. Tokenize biomedical text.
2. Detect known prefixes.
3. Detect kno

In [6]:
#CO2_AT3.2

import re
import pandas as pd
from collections import defaultdict

words = [
    "replayed",
    "unhappier",
    "disconnected",
    "players",
    "restarting",
    "unreadable"
]

correct_analyses = {
    "replayed": ["re", "play", "ed"],
    "unhappier": ["un", "happy", "er"],
    "disconnected": ["dis", "connect", "ed"],
    "players": ["", "play", "er", "s"],
    "restarting": ["re", "start", "ing"],
    "unreadable": ["un", "read", "able"]
}

single_affix_parser = {
    "re": "PREFIX",
    "un": "PREFIX",
    "dis": "PREFIX",
    "ed": "SUFFIX",
    "er": "SUFFIX",
    "s": "SUFFIX",
    "ing": "SUFFIX",
    "able": "SUFFIX"
}

def original_parser(word):
    for affix in ["re", "un", "dis"]:
        if word.startswith(affix):
            root = word[len(affix):]
            return [affix, root]

    for affix in ["ed", "er", "s", "ing", "able"]:
        if word.endswith(affix):
            root = word[:-len(affix)]
            return [root, affix]

    return [word]

def corrected_parser(word):
    prefixes = ["un", "dis", "re"]
    suffixes = ["able", "ing", "ied", "ed", "er", "s"]

    prefix_list = []
    suffix_list = []
    remaining = word

    changed = True

    while changed:
        changed = False

        for prefix in sorted(prefixes, key=len, reverse=True):
            if remaining.startswith(prefix):
                prefix_list.append(prefix)
                remaining = remaining[len(prefix):]
                changed = True
                break

    changed = True

    while changed:
        changed = False

        for suffix in sorted(suffixes, key=len, reverse=True):
            if remaining.endswith(suffix) and len(remaining) > len(suffix):
                suffix_list.insert(0, suffix)
                remaining = remaining[:-len(suffix)]
                changed = True
                break

    if remaining.endswith("i") and suffix_list and suffix_list[0] == "er":
        remaining = remaining[:-1] + "y"

    return prefix_list + [remaining] + suffix_list

print("\nORIGINAL PARSER OUTPUT")

original_results = {}

for word in words:
    result = original_parser(word)
    original_results[word] = result
    print(f"{word:15} -> {result}")

print("\nCORRECTED FST PARSER OUTPUT")

corrected_results = {}

for word in words:
    result = corrected_parser(word)
    corrected_results[word] = result
    print(f"{word:15} -> {result}")

print("\nFST TRANSITIONS")

transitions = defaultdict(dict)

for affix, affix_type in single_affix_parser.items():
    transitions["START"][affix] = affix_type

for state in ["PREFIX", "ROOT", "SUFFIX"]:
    transitions[state]["re"] = "PREFIX"
    transitions[state]["un"] = "PREFIX"
    transitions[state]["dis"] = "PREFIX"
    transitions[state]["ed"] = "SUFFIX"
    transitions[state]["er"] = "SUFFIX"
    transitions[state]["s"] = "SUFFIX"
    transitions[state]["ing"] = "SUFFIX"
    transitions[state]["able"] = "SUFFIX"
    transitions[state]["ROOT"] = "ACCEPT"

for state, values in transitions.items():
    print(f"{state}: {values}")

print("\nPARSER COMPARISON")

for word in words:
    expected = correct_analyses[word]
    original = original_results[word]
    corrected = corrected_results[word]

    print(f"\nWord: {word}")
    print(f"Expected : {expected}")
    print(f"Original : {original}")
    print(f"Corrected: {corrected}")

def calculate_accuracy(results, expected):
    correct = 0

    for word in expected:
        if results[word] == expected[word]:
            correct += 1

    return correct / len(expected) * 100

original_accuracy = calculate_accuracy(original_results, correct_analyses)
corrected_accuracy = calculate_accuracy(corrected_results, correct_analyses)

print("\nACCURACY")
print(f"Original Accuracy : {original_accuracy:.2f}%")
print(f"Corrected Accuracy: {corrected_accuracy:.2f}%")

sentiment_url = (
    "https://raw.githubusercontent.com/zeerakhan/"
    "Sentiment-Analysis-on-Sentiment140-Dataset/master/"
    "training.1600000.processed.noemoticon.csv"
)

try:
    sentiment = pd.read_csv(
        sentiment_url,
        encoding="latin-1",
        header=None,
        names=["sentiment", "id", "date", "query", "user", "text"],
        nrows=5000
    )

    text = " ".join(sentiment["text"].astype(str))

    dataset_words = re.findall(r"[A-Za-z]+", text.lower())

    parsed_dataset = []

    for token in dataset_words:
        parsed_dataset.append({
            "word": token,
            "analysis": corrected_parser(token)
        })

    parsed_df = pd.DataFrame(parsed_dataset)

    print("\nCORRECTED PARSER ON SENTIMENT140")
    print(parsed_df.head(30).to_string(index=False))

except Exception as e:
    print("\nSentiment140 dataset could not be loaded:", e)

print("\nCOMPUTATIONAL COMPLEXITY")
print("Prefix matching: O(P)")
print("Suffix matching: O(S)")
print("For a word of length n, repeated transitions are O(n).")
print("For N words, total parsing complexity is approximately O(N*n).")
print("Memory complexity is O(P + S) for the transition tables.")


ORIGINAL PARSER OUTPUT
replayed        -> ['re', 'played']
unhappier       -> ['un', 'happier']
disconnected    -> ['dis', 'connected']
players         -> ['player', 's']
restarting      -> ['re', 'starting']
unreadable      -> ['un', 'readable']

CORRECTED FST PARSER OUTPUT
replayed        -> ['re', 'play', 'ed']
unhappier       -> ['un', 'happy', 'er']
disconnected    -> ['dis', 'connect', 'ed']
players         -> ['play', 'er', 's']
restarting      -> ['re', 'start', 'ing']
unreadable      -> ['un', 're', 'ad', 'able']

FST TRANSITIONS
START: {'re': 'PREFIX', 'un': 'PREFIX', 'dis': 'PREFIX', 'ed': 'SUFFIX', 'er': 'SUFFIX', 's': 'SUFFIX', 'ing': 'SUFFIX', 'able': 'SUFFIX'}
PREFIX: {'re': 'PREFIX', 'un': 'PREFIX', 'dis': 'PREFIX', 'ed': 'SUFFIX', 'er': 'SUFFIX', 's': 'SUFFIX', 'ing': 'SUFFIX', 'able': 'SUFFIX', 'ROOT': 'ACCEPT'}
ROOT: {'re': 'PREFIX', 'un': 'PREFIX', 'dis': 'PREFIX', 'ed': 'SUFFIX', 'er': 'SUFFIX', 's': 'SUFFIX', 'ing': 'SUFFIX', 'able': 'SUFFIX', 'ROOT': 'ACCEPT'}
S

In [7]:
#CO2_AT3.3

import re
import pandas as pd
from nltk.stem import PorterStemmer

ps = PorterStemmer()

data = pd.DataFrame({
    "Text": [
        "The researchers studied treatments for patients.",
        "Doctors are treating patients with improved medicines.",
        "The study analyzed different medical procedures.",
        "Researchers developed better treatments and therapies."
    ]
})

def original_processing(text):
    return ps.stem(text)

def tokenize(text):
    return re.findall(r"\b[a-zA-Z]+\b", text.lower())

def corrected_processing(text):
    tokens = tokenize(text)
    stems = [ps.stem(token) for token in tokens]
    return tokens, stems

data["Original_Output"] = data["Text"].apply(original_processing)

data["Tokens"] = data["Text"].apply(
    lambda x: corrected_processing(x)[0]
)

data["Stemmed_Tokens"] = data["Text"].apply(
    lambda x: corrected_processing(x)[1]
)

print("\nORIGINAL PROGRAM OUTPUT")
print(data[["Text", "Original_Output"]].to_string(index=False))

print("\nCORRECTED PROGRAM OUTPUT")
print(
    data[
        ["Text", "Tokens", "Stemmed_Tokens"]
    ].to_string(index=False)
)

print("\nPROBLEMATIC STEMMING CASES")

problematic_cases = [
    ("studies", "studi", "study", "inflectional"),
    ("studied", "studi", "study", "inflectional"),
    ("studying", "studi", "study", "inflectional"),
    ("treatment", "treatment", "treat", "derivational"),
    ("treatments", "treatment", "treat", "derivational + inflectional"),
    ("treating", "treat", "treat", "inflectional"),
    ("treated", "treat", "treat", "inflectional"),
    ("connectivity", "connect", "connect", "derivational"),
    ("connected", "connect", "connect", "inflectional"),
    ("connecting", "connect", "connect", "inflectional"),
    ("connection", "connect", "connect", "derivational"),
    ("organizations", "organ", "organize", "derivational + inflectional"),
    ("organization", "organ", "organize", "derivational"),
    ("organizer", "organ", "organize", "derivational"),
    ("organized", "organ", "organize", "inflectional"),
    ("organizing", "organ", "organize", "inflectional"),
    ("procedures", "procedur", "procedure", "inflectional"),
    ("procedural", "procedur", "procedure", "derivational"),
    ("patients", "patient", "patient", "inflectional"),
    ("therapies", "therapi", "therapy", "inflectional")
]

for word, stem, expected, affix_type in problematic_cases:
    print(
        f"{word:18} -> Stem: {stem:15} "
        f"Expected: {expected:12} Type: {affix_type}"
    )

print("\nAG NEWS DATASET")

agnews_url = (
    "https://raw.githubusercontent.com/mhjabreel/"
    "CharCnn_Keras/master/data/ag_news_csv/train.csv"
)

try:
    agnews = pd.read_csv(
        agnews_url,
        header=None,
        names=["Class", "Title", "Description"],
        nrows=1000
    )

    agnews["Text"] = (
        agnews["Title"].astype(str)
        + " "
        + agnews["Description"].astype(str)
    )

    agnews["Tokens"] = agnews["Text"].apply(tokenize)

    agnews["Stemmed_Tokens"] = agnews["Tokens"].apply(
        lambda tokens: [ps.stem(token) for token in tokens]
    )

    print(
        agnews[
            ["Text", "Tokens", "Stemmed_Tokens"]
        ].head(10).to_string(index=False)
    )

except Exception as e:
    print("\nAG News dataset could not be loaded:", e)

print("\nCORRECTED PIPELINE")
print("Raw Text")
print("   ↓")
print("Sentence/Text Input")
print("   ↓")
print("Tokenization")
print("   ↓")
print("Normalization")
print("   ↓")
print("Stemming")
print("   ↓")
print("Processed Tokens")
print("   ↓")
print("Feature Extraction")


ORIGINAL PROGRAM OUTPUT
                                                  Text                                        Original_Output
      The researchers studied treatments for patients.       the researchers studied treatments for patients.
Doctors are treating patients with improved medicines. doctors are treating patients with improved medicines.
      The study analyzed different medical procedures.       the study analyzed different medical procedures.
Researchers developed better treatments and therapies. researchers developed better treatments and therapies.

CORRECTED PROGRAM OUTPUT
                                                  Text                                                        Tokens                                       Stemmed_Tokens
      The researchers studied treatments for patients.        [the, researchers, studied, treatments, for, patients]      [the, research, studi, treatment, for, patient]
Doctors are treating patients with improved medicines. [doc

In [8]:
#CO2_AT3.4

import nltk

nltk.download("wordnet", quiet=True)

from nltk.corpus import wordnet as wn

words = [
    "cars",
    "boxes",
    "cities",
    "children",
    "books"
]

irregular_plurals = {
    "children": "child",
    "men": "man",
    "women": "woman",
    "people": "person",
    "mice": "mouse",
    "geese": "goose",
    "teeth": "tooth",
    "feet": "foot"
}

def original_parser(word):
    if word.endswith("s"):
        return word[:-2], "Plural Noun"
    else:
        return word, "Singular"

def corrected_parser(word):
    if word in irregular_plurals:
        return irregular_plurals[word], "Plural Noun"

    if word.endswith("ies") and len(word) > 3:
        return word[:-3] + "y", "Plural Noun"

    if word.endswith("ses") and len(word) > 3:
        return word[:-2], "Plural Noun"

    if word.endswith("xes") or word.endswith("zes"):
        return word[:-2], "Plural Noun"

    if word.endswith("ches") or word.endswith("shes"):
        return word[:-2], "Plural Noun"

    if word.endswith("es") and len(word) > 2:
        return word[:-2], "Plural Noun"

    if word.endswith("s") and not word.endswith("ss"):
        return word[:-1], "Plural Noun"

    return word, "Singular"

print("\nORIGINAL OUTPUT")

for word in words:
    print(word, "->", original_parser(word))

print("\nCORRECTED OUTPUT")

for word in words:
    print(word, "->", corrected_parser(word))

print("\nWORDNET VALIDATION")

for word in words:
    lemma, category = corrected_parser(word)

    synsets = wn.synsets(word, pos=wn.NOUN)
    lemma_names = set()

    for synset in synsets:
        for lemma_obj in synset.lemmas():
            lemma_names.add(lemma_obj.name())

    print(
        f"{word:12} -> {lemma:12} "
        f"WordNet entries: {list(lemma_names)[:5]}"
    )

print("\nMORPHOLOGICAL RULES")

rules = {
    "Regular plural": "car + s -> cars",
    "-es plural": "box + es -> boxes",
    "-ies plural": "city -> cities",
    "Irregular plural": "child -> children"
}

for rule, transformation in rules.items():
    print(f"{rule:20}: {transformation}")

print("\nLOGICAL ERRORS IN ORIGINAL PROGRAM")
print("1. word[:-2] removes two characters for every word ending in s.")
print("2. Regular -s plurals require removal of only one character.")
print("3. -es plurals require removal of -es.")
print("4. -ies plurals require conversion of -ies to -y.")
print("5. Irregular plurals cannot be handled by suffix stripping.")
print("6. Words ending in ss may be incorrectly classified.")
print("7. The parser has no lexical validation.")
print("8. The parser does not distinguish plural suffixes from lexical s.")
print("9. The parser contains no state transitions for multiple rules.")
print("10. The parser assumes every word ending in s is plural.")


ORIGINAL OUTPUT
cars -> ('ca', 'Plural Noun')
boxes -> ('box', 'Plural Noun')
cities -> ('citi', 'Plural Noun')
children -> ('children', 'Singular')
books -> ('boo', 'Plural Noun')

CORRECTED OUTPUT
cars -> ('car', 'Plural Noun')
boxes -> ('box', 'Plural Noun')
cities -> ('city', 'Plural Noun')
children -> ('child', 'Plural Noun')
books -> ('book', 'Plural Noun')

WORDNET VALIDATION
cars         -> car          WordNet entries: ['railcar', 'railway_car', 'gondola', 'cable_car', 'machine']
boxes        -> box          WordNet entries: ['corner', 'box', 'box_seat', 'boxwood', 'boxful']
cities       -> city         WordNet entries: ['metropolis', 'city', 'urban_center']
children     -> child        WordNet entries: ['nipper', 'youngster', 'shaver', 'fry', 'tike']
books        -> book         WordNet entries: ['Book', 'account_book', 'Holy_Writ', 'Holy_Scripture', 'Word']

MORPHOLOGICAL RULES
Regular plural      : car + s -> cars
-es plural          : box + es -> boxes
-ies plural        

In [9]:
#CO2_AT3.5

import re
import time
import pandas as pd

from nltk.stem import PorterStemmer
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

stemmer = PorterStemmer()

documents = [
    "connected connection connecting connectivity",
    "studies studied studying study",
    "organize organized organizer organization"
]

print("\nORIGINAL PIPELINE")

original_vectorizer = CountVectorizer()

start = time.time()

X_original = original_vectorizer.fit_transform(documents)

original_features = [
    stemmer.stem(word)
    for word in original_vectorizer.get_feature_names_out()
]

original_time = time.time() - start

print("Original Vocabulary:")
print(original_vectorizer.get_feature_names_out())

print("\nStemmed Vocabulary After Feature Extraction:")
print(original_features)

print(
    "\nOriginal Vocabulary Size:",
    len(original_vectorizer.get_feature_names_out())
)

print(
    "Vocabulary Size After Stemming Features:",
    len(set(original_features))
)

def normalize_text(text):
    tokens = re.findall(r"\b[a-zA-Z]+\b", text.lower())
    return " ".join(stemmer.stem(token) for token in tokens)

print("\nCORRECTED PIPELINE")

normalized_documents = [
    normalize_text(document)
    for document in documents
]

corrected_vectorizer = CountVectorizer()

start = time.time()

X_corrected = corrected_vectorizer.fit_transform(
    normalized_documents
)

corrected_time = time.time() - start

print("\nNormalized Documents:")

for original, normalized in zip(documents, normalized_documents):
    print(f"\nOriginal : {original}")
    print(f"Normalized: {normalized}")

print("\nCorrected Vocabulary:")
print(corrected_vectorizer.get_feature_names_out())

print(
    "\nCorrected Vocabulary Size:",
    len(corrected_vectorizer.get_feature_names_out())
)

print("\nPROCESSING TIME")
print(f"Original: {original_time:.6f} seconds")
print(f"Corrected: {corrected_time:.6f} seconds")

print("\n10 MORPHOLOGICAL NORMALIZATION EXAMPLES")

examples = [
    ("connected", "connect"),
    ("connection", "connect"),
    ("connecting", "connect"),
    ("connectivity", "connect"),
    ("studies", "studi"),
    ("studied", "studi"),
    ("studying", "studi"),
    ("organized", "organ"),
    ("organizer", "organ"),
    ("organization", "organ")
]

for original, normalized in examples:
    print(f"{original:20} -> {normalized}")

print("\n20 NEWSGROUPS DATASET")

categories = [
    "sci.med",
    "sci.space",
    "comp.graphics",
    "rec.sport.baseball"
]

train_data = fetch_20newsgroups(
    subset="train",
    categories=categories,
    remove=("headers", "footers", "quotes")
)

test_data = fetch_20newsgroups(
    subset="test",
    categories=categories,
    remove=("headers", "footers", "quotes")
)

X_train_text = train_data.data
X_test_text = test_data.data

y_train = train_data.target
y_test = test_data.target

print("Training Documents:", len(X_train_text))
print("Testing Documents:", len(X_test_text))

print("\nBASELINE MODEL")

baseline_vectorizer = CountVectorizer(
    stop_words="english",
    max_features=20000
)

start = time.time()

X_train_base = baseline_vectorizer.fit_transform(X_train_text)
X_test_base = baseline_vectorizer.transform(X_test_text)

baseline_model = LogisticRegression(
    max_iter=1000
)

baseline_model.fit(X_train_base, y_train)

baseline_predictions = baseline_model.predict(X_test_base)

baseline_accuracy = accuracy_score(
    y_test,
    baseline_predictions
)

baseline_time = time.time() - start

print(
    "Baseline Vocabulary Size:",
    len(baseline_vectorizer.get_feature_names_out())
)

print(
    f"Baseline Classification Accuracy: "
    f"{baseline_accuracy * 100:.2f}%"
)

print(
    f"Baseline Processing Time: "
    f"{baseline_time:.4f} seconds"
)

print("\nCORRECTED MORPHOLOGICAL MODEL")

start = time.time()

X_train_normalized = [
    normalize_text(text)
    for text in X_train_text
]

X_test_normalized = [
    normalize_text(text)
    for text in X_test_text
]

corrected_vectorizer = CountVectorizer(
    stop_words="english",
    max_features=20000
)

X_train_corrected = corrected_vectorizer.fit_transform(
    X_train_normalized
)

X_test_corrected = corrected_vectorizer.transform(
    X_test_normalized
)

corrected_model = LogisticRegression(
    max_iter=1000
)

corrected_model.fit(
    X_train_corrected,
    y_train
)

corrected_predictions = corrected_model.predict(
    X_test_corrected
)

corrected_accuracy = accuracy_score(
    y_test,
    corrected_predictions
)

corrected_time = time.time() - start

print(
    "Corrected Vocabulary Size:",
    len(corrected_vectorizer.get_feature_names_out())
)

print(
    f"Corrected Classification Accuracy: "
    f"{corrected_accuracy * 100:.2f}%"
)

print(
    f"Corrected Processing Time: "
    f"{corrected_time:.4f} seconds"
)

print("\nFINAL COMPARISON")

comparison = pd.DataFrame({
    "Metric": [
        "Vocabulary Size",
        "Classification Accuracy (%)",
        "Processing Time (seconds)"
    ],
    "Before Correction": [
        len(baseline_vectorizer.get_feature_names_out()),
        baseline_accuracy * 100,
        baseline_time
    ],
    "After Correction": [
        len(corrected_vectorizer.get_feature_names_out()),
        corrected_accuracy * 100,
        corrected_time
    ]
})

print(comparison.to_string(index=False))

print("\nMORPHOLOGICAL NORMALIZATION COMPARISON")

comparison_words = [
    "connected",
    "connection",
    "connecting",
    "connectivity",
    "studies",
    "studied",
    "studying",
    "study",
    "organize",
    "organized",
    "organizer",
    "organization"
]

for word in comparison_words:
    print(
        f"{word:20} -> "
        f"{stemmer.stem(word)}"
    )

print("\nPIPELINE")

print("""
BEFORE:
Raw Documents
    |
    v
CountVectorizer
    |
    v
Feature Extraction
    |
    v
Vocabulary
    |
    v
Stemming

AFTER:
Raw Documents
    |
    v
Tokenization
    |
    v
Normalization
    |
    v
Stemming
    |
    v
CountVectorizer
    |
    v
Normalized Vocabulary
    |
    v
Machine Learning Classifier
""")


ORIGINAL PIPELINE
Original Vocabulary:
['connected' 'connecting' 'connection' 'connectivity' 'organization'
 'organize' 'organized' 'organizer' 'studied' 'studies' 'study' 'studying']

Stemmed Vocabulary After Feature Extraction:
['connect', 'connect', 'connect', 'connect', 'organ', 'organ', 'organ', 'organ', 'studi', 'studi', 'studi', 'studi']

Original Vocabulary Size: 12
Vocabulary Size After Stemming Features: 3

CORRECTED PIPELINE

Normalized Documents:

Original : connected connection connecting connectivity
Normalized: connect connect connect connect

Original : studies studied studying study
Normalized: studi studi studi studi

Original : organize organized organizer organization
Normalized: organ organ organ organ

Corrected Vocabulary:
['connect' 'organ' 'studi']

Corrected Vocabulary Size: 3

PROCESSING TIME
Original: 0.030168 seconds
Corrected: 0.004063 seconds

10 MORPHOLOGICAL NORMALIZATION EXAMPLES
connected            -> connect
connection           -> connect
connecti